# Option 2 — Symbolic Conditioned MIDI Generation

**Task**: Given a 4-second MIDI prefix, autoregressively generate a 4-second continuation using a small GPT-2 Transformer trained on REMI tokens from MAESTRO v3.0.0.

**Pipeline**: Setup → Download MAESTRO → Build REMI token cache → Build windows → Train → Generate → Evaluate

## 1. Environment Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab:', IN_COLAB)

if IN_COLAB:
    REPO_URL = 'https://github.com/archerop/CSE_253.git'
    REPO_DIR = Path('/content/CSE_253')
    BRANCH   = 'mao_dev'
    if not REPO_DIR.exists():
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        result = !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull --ff-only origin {BRANCH}
        print('\n'.join(result))
        # Flush stale cached modules so re-imports read the updated files
        stale = [k for k in sys.modules if k == 'app' or k.startswith('app.')]
        for k in stale:
            del sys.modules[k]
        importlib.invalidate_caches()
        if stale:
            print(f'Flushed {len(stale)} cached module(s)')
    ASSIGNMENT_ROOT = REPO_DIR / 'assignment2'
else:
    ASSIGNMENT_ROOT = Path('__file__').resolve().parent if '__file__' in dir() else Path('.').resolve()
    for p in [ASSIGNMENT_ROOT] + list(ASSIGNMENT_ROOT.parents):
        if (p / 'app').exists():
            ASSIGNMENT_ROOT = p
            break

print('Assignment root:', ASSIGNMENT_ROOT)
os.chdir(ASSIGNMENT_ROOT)
if str(ASSIGNMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_ROOT))

## 2. Install Dependencies

In [ ]:
import importlib, subprocess

def need(pkg):
    return importlib.util.find_spec(pkg) is None

pkgs = [p for p in ['pretty_midi', 'torch', 'pandas', 'tqdm', 'scipy', 'miditok', 'symusic', 'transformers'] if need(p)]
if pkgs:
    print('Installing:', pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
else:
    print('All dependencies already installed.')

## 3. Download MAESTRO MIDI Dataset

In [ ]:
import urllib.request, zipfile

MAESTRO_ROOT = ASSIGNMENT_ROOT / 'data' / 'maestro-v3.0.0'

if (MAESTRO_ROOT / 'maestro-v3.0.0.csv').exists():
    print('MAESTRO already present at', MAESTRO_ROOT)
else:
    URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
    ZIP = ASSIGNMENT_ROOT / 'data' / 'downloads' / 'maestro-v3.0.0-midi.zip'
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists():
        print('Downloading MAESTRO MIDI (~57 MB)...')
        urllib.request.urlretrieve(URL, ZIP)
    print('Extracting...')
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ASSIGNMENT_ROOT / 'data')
    print('Done.')

print(f'MIDI files: {len(list(MAESTRO_ROOT.rglob("*.midi")))}')

## 4. Hyperparameters & Imports

Hyperparameters are defined directly here so you can tune them without editing `app/shared/config.py`. The values are pushed into the config module before the option2 modules are imported, so downstream defaults see the same values.

In [ ]:
# ── Hyperparameters (edit these — notebook is the source of truth) ──────────
# Sequence / window settings
OPTION2_PREFIX_SECONDS       = 4.0    # input prefix duration (sec)
OPTION2_CONTINUATION_SECONDS = 4.0    # continuation duration to generate (sec)
OPTION2_STRIDE_SECONDS       = 2.0    # window stride (sec)
OPTION2_FRAME_RATE           = 40.0   # piano-roll fps used by evaluation

OPTION2_PREFIX_MAX_LEN = 256          # max tokens for prefix
OPTION2_CONT_MAX_LEN   = 256          # max tokens for continuation
OPTION2_MAX_SEQ_LEN    = 512          # PREFIX + CONT fed to model

# Optimization
OPTION2_BATCH_SIZE    = 32
OPTION2_LEARNING_RATE = 3e-4
OPTION2_WEIGHT_DECAY  = 1e-4
OPTION2_MAX_EPOCHS    = 20      # loss was still declining at epoch 15 — train to convergence
OPTION2_PATIENCE      = 8       # give more room before early-stopping
OPTION2_DROPOUT       = 0.1

# Transformer architecture — ~0.88M params, T4-safe
# Param ladder: d=128→0.88M | d=192→1.91M | d=256→4.92M
OPTION2_D_MODEL         = 128
OPTION2_NHEAD           = 4
OPTION2_NUM_LAYERS      = 4
OPTION2_DIM_FEEDFORWARD = 512

# GPT-2 architecture
OPTION2_GPT2_N_LAYER = 6
OPTION2_GPT2_N_HEAD  = 8
OPTION2_GPT2_N_EMBD  = 384

# RNN architecture (LSTM / GRU)
OPTION2_HIDDEN_SIZE = 256
OPTION2_RNN_LAYERS  = 2

# ── Generation sampling parameters ──────────────────────────────────────────
GENERATION_TEMPERATURE = 0.8
GENERATION_TOP_K       = 10

# Push the notebook values into the config module BEFORE importing the option2
# modules, so any default args / module-level constants in those modules
# bind to the notebook values. We also flush any cached option2 modules so
# they are re-imported fresh and re-bind their `from config import ...` names.
import sys
import app.shared.config as _cfg
for _k, _v in list(locals().items()):
    if _k.startswith("OPTION2_"):
        setattr(_cfg, _k, _v)
for _k in list(sys.modules):
    if _k.startswith("app.option2"):
        del sys.modules[_k]

# ── Imports ─────────────────────────────────────────────────────────────────
import json, pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
import pretty_midi
from tqdm.auto import tqdm

# Paths only — hyperparameters are defined above, not imported.
from app.shared.config import (
    MAESTRO_ROOT, OPTION2_CACHE_DIR, OPTION2_OUTPUT_DIR, CHECKPOINT_DIR,
)
from app.shared.metadata import load_maestro_metadata, validate_maestro_paths
from app.option2.symbolic_dataset import (
    build_tokenizer, precache_tokens, build_token_window_index, SymbolicDataset, get_datasets,
)
from app.option2.symbolic_models import build_model, generate_tokens, CopyLastPatternBaseline
from app.option2.symbolic_train import train, load_best_checkpoint
from app.option2.symbolic_generate import (
    extract_prefix, generate_conditioned, save_symbolic_conditioned, tokens_to_pianoroll,
)
from app.option2.symbolic_eval import evaluate_generation, evaluate_token_generation, print_metrics

print("Hyperparameters set; imports OK")

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

# ── Model selection ──────────────────────────────────────────────────────────
# Change MODEL_TYPE to switch architectures. All share the same token pipeline.
#   'lstm'        ~1.2M params  (fast, simple, good baseline)
#   'gru'         ~1.0M params  (slightly smaller than LSTM)
#   'transformer' ~0.9M params  (causal, token-level)
#   'gpt2'        ~11M params   (deepest, best quality, slower to train)
MODEL_TYPE = 'transformer'
# ─────────────────────────────────────────────────────────────────────────────
print('Model type:', MODEL_TYPE)

## 5. Google Drive Setup & Output Directory
On Colab, Google Drive is mounted and **all outputs** (checkpoint, loss curve, eval results,
piano-roll, generated MIDI) are written to `MyDrive/CSE253/option2/`.
On a local machine the same `LOG_DIR` variable falls back to `logs/option2/` inside the repo.

In [ ]:
from pathlib import Path
from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/CSE253') / 'option2'
else:
    BASE_DIR = ASSIGNMENT_ROOT / 'logs' / 'option2'

# One folder per training run: option2/{MODEL_TYPE}_{timestamp}/
RUN_DIR = BASE_DIR / f'{MODEL_TYPE}_{RUN_TIMESTAMP}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = RUN_DIR / f'option2_{MODEL_TYPE}_best.pt'
print('BASE_DIR   →', BASE_DIR)
print('RUN_DIR    →', RUN_DIR)
print('Checkpoint →', CKPT_PATH)

## 6. Build REMI Tokenizer & Pre-cache Tokens
Tokenizes all 1,276 MIDI files once using the REMI tokenizer and saves per-file `.pkl` caches. Safe to re-run — skips existing files.

In [ ]:
tokenizer = build_tokenizer()
print(f'Tokenizer ready. vocab_size={tokenizer.vocab_size}  PAD={tokenizer["PAD_None"]}  BOS={tokenizer["BOS_None"]}  EOS={tokenizer["EOS_None"]}')
precache_tokens(tokenizer)

## 7. Build Token Window Index
Slides a window of `PREFIX_MAX_LEN + CONT_MAX_LEN = 512` tokens over each MIDI's token sequence with stride 128.

In [ ]:
OPTION2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

_window_kw = dict(
    prefix_seconds=OPTION2_PREFIX_SECONDS,
    continuation_seconds=OPTION2_CONTINUATION_SECONDS,
    stride_seconds=OPTION2_STRIDE_SECONDS,
)

train_windows = build_token_window_index('train',      tokenizer, **_window_kw, cache_path=OPTION2_CACHE_DIR / 'train_token_windows.pkl')
val_windows   = build_token_window_index('validation', tokenizer, **_window_kw, cache_path=OPTION2_CACHE_DIR / 'val_token_windows.pkl')
test_windows  = build_token_window_index('test',       tokenizer, **_window_kw, cache_path=OPTION2_CACHE_DIR / 'test_token_windows.pkl')

print(f'Windows — train: {len(train_windows):,}  val: {len(val_windows):,}  test: {len(test_windows):,}')

## 8. Dataset & DataLoader

In [ ]:
train_ds = SymbolicDataset(train_windows, tokenizer)
val_ds   = SymbolicDataset(val_windows,   tokenizer)

prefix, cont = train_ds[0]
print(f'prefix shape: {tuple(prefix.shape)}  dtype: {prefix.dtype}')
print(f'cont shape:   {tuple(cont.shape)}   dtype: {cont.dtype}')
print(f'non-pad prefix tokens: {(prefix != tokenizer["PAD_None"]).sum().item()}')

In [ ]:
PIN_MEMORY  = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if IN_COLAB else 4

train_loader = DataLoader(train_ds, batch_size=OPTION2_BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=OPTION2_BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')

## 9. Model

All models use REMI tokenization (vocab 347) and are trained with the same teacher-forcing cross-entropy loss. Switch `MODEL_TYPE` in the device cell above to compare.

| MODEL_TYPE | Architecture | Params | Notes |
|------------|-------------|--------|-------|
| `lstm` | 2-layer LSTM, hidden=256 | ~1.2M | Fast, strong sequential baseline |
| `gru` | 2-layer GRU, hidden=256 | ~1.0M | Slightly lighter than LSTM |
| `transformer` | 4-layer causal Transformer, d=128 | ~0.9M | Lightest, explicit position encoding |
| `gpt2` | 6-layer GPT-2, d=384 | ~11M | Best quality, ~10× slower |

In [ ]:
model = build_model(MODEL_TYPE, vocab_size=tokenizer.vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_TYPE}  |  Parameters: {n_params:,} ({n_params/1e6:.2f}M)')

## 10. Training

In [ ]:
history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    checkpoint_path=CKPT_PATH,
    max_epochs=OPTION2_MAX_EPOCHS,
    patience=OPTION2_PATIENCE,
    lr=OPTION2_LEARNING_RATE,
    weight_decay=OPTION2_WEIGHT_DECAY,
)

with open(RUN_DIR / 'train_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Training complete. All outputs in:', RUN_DIR)

## 11. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='Train')
plt.plot(epochs, history['val_loss'],   label='Val')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title(f'Option 2 Training Loss ({MODEL_TYPE}) — {RUN_TIMESTAMP}')
plt.legend()
plt.tight_layout()
loss_curve_path = RUN_DIR / f'loss_curve_{MODEL_TYPE}.png'
plt.savefig(loss_curve_path, dpi=150)
plt.show()
print('Loss curve saved to', loss_curve_path)

## 12. Generate `symbolic_conditioned.mid`

In [ ]:
df = load_maestro_metadata(MAESTRO_ROOT)
df = validate_maestro_paths(df)
test_df = df[(df['split'] == 'test') & df['midi_exists']].sort_values('duration').reset_index(drop=True)
row = test_df.iloc[len(test_df) // 2]
MIDI_PATH = row['midi_path']
print(f"Using: {row['composer']} — {row['title']}  ({row['duration']:.1f}s)")

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/CSE253/option2/transformer_20260528_052335/option2_transformer_best.pt"
model = load_best_checkpoint(build_model(MODEL_TYPE, vocab_size=tokenizer.vocab_size), MODEL_PATH, DEVICE)

output_path = save_symbolic_conditioned(
    prefix_midi_path=MIDI_PATH,
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    output_path=RUN_DIR / 'symbolic_conditioned.mid',
    temperature=GENERATION_TEMPERATURE,
    top_k=GENERATION_TOP_K,
)
print('Output saved to:', output_path)

## 13. Evaluation — Model vs Baseline

In [ ]:
import symusic
from app.option2.symbolic_dataset import build_token_window_index, _trim_score
from app.option2.symbolic_eval import evaluate_dataset, print_dataset_metrics

test_windows = build_token_window_index('test', tokenizer,
                   cache_path=OPTION2_CACHE_DIR / 'test_token_windows.pkl')

# One representative window per MIDI file (middle window)
by_file = {}
for w in test_windows:
    by_file.setdefault(w[0], []).append(w)
test_samples = [ws[len(ws) // 2] for ws in by_file.values()]
print(f'Evaluating on {len(test_samples)} test pieces')

In [ ]:
## ── Generation diagnostics (run this to debug blank outputs) ────────────────
# Checks: (1) how many tokens are generated before EOS, (2) decode success rate
import warnings, traceback

print(f"Sampling with temperature={GENERATION_TEMPERATURE}, top_k={GENERATION_TOP_K}")
print("Running diagnostics on 10 random test pieces...")

EOS_ID = tokenizer["EOS_None"]
PAD_ID = tokenizer["PAD_None"]

decode_ok = 0
eos_positions = []
non_pad_counts = []

diag_samples = test_samples[:10]
model.eval()
for midi_path, *_ in diag_samples:
    prefix_ids = extract_prefix(midi_path, tokenizer)
    cont_ids   = generate_conditioned(model, prefix_ids, OPTION2_CONT_MAX_LEN, DEVICE,
                                      temperature=GENERATION_TEMPERATURE, top_k=GENERATION_TOP_K)
    toks = cont_ids.tolist()
    non_pad = [t for t in toks if t != PAD_ID]
    non_pad_counts.append(len(non_pad))

    # Where does EOS appear?
    eos_pos = next((i for i, t in enumerate(toks) if t == EOS_ID), -1)
    eos_positions.append(eos_pos)

    # Try decode
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        roll = tokens_to_pianoroll(toks, tokenizer)
        if not w:
            decode_ok += 1

print(f"\n  Decode successes : {decode_ok}/10")
print(f"  Mean non-PAD tokens generated : {sum(non_pad_counts)/len(non_pad_counts):.1f}")
print(f"  EOS positions (index, -1=none): {eos_positions}")
print(f"  Avg note density in decoded rolls: "
      f"{sum(tokens_to_pianoroll(generate_conditioned(model, extract_prefix(s[0], tokenizer), OPTION2_CONT_MAX_LEN, DEVICE, GENERATION_TEMPERATURE, GENERATION_TOP_K).tolist(), tokenizer).sum(axis=1).mean() for s in diag_samples) / len(diag_samples):.4f}")

In [ ]:
model.eval()
gen_rolls, gt_rolls, baseline_rolls = [], [], []

for midi_path, start_tok, prefix_end_tok, cont_end_tok in tqdm(test_samples):
    # Ground truth continuation tokens → piano-roll
    score   = symusic.Score(midi_path)
    cont_sm = _trim_score(score, OPTION2_PREFIX_SECONDS,
                          OPTION2_PREFIX_SECONDS + OPTION2_CONTINUATION_SECONDS)
    gt_seqs = tokenizer.encode(cont_sm)
    gt_ids  = gt_seqs[0].ids if gt_seqs else []
    gt_roll = tokens_to_pianoroll(gt_ids, tokenizer)

    # Model generation (lower temperature, narrower top-k)
    prefix_ids = extract_prefix(midi_path, tokenizer)
    cont_ids   = generate_conditioned(model, prefix_ids, OPTION2_CONT_MAX_LEN, DEVICE,
                                      temperature=GENERATION_TEMPERATURE,
                                      top_k=GENERATION_TOP_K)
    gen_roll   = tokens_to_pianoroll(cont_ids.tolist(), tokenizer)

    # Baseline: cycle last prefix tokens
    bl_ids  = CopyLastPatternBaseline().generate(
                  prefix_ids.unsqueeze(0), OPTION2_CONT_MAX_LEN).squeeze(0)
    bl_roll = tokens_to_pianoroll(bl_ids.tolist(), tokenizer)

    gen_rolls.append(gen_roll)
    gt_rolls.append(gt_roll)
    baseline_rolls.append(bl_roll)

print(f'Generated {len(gen_rolls)} continuations')

In [ ]:
import json

print(f'=== Model ({MODEL_TYPE}) ===')
model_metrics = evaluate_dataset(gen_rolls, gt_rolls)
print_dataset_metrics(model_metrics)

print('=== Baseline (copy last pattern) ===')
baseline_metrics = evaluate_dataset(baseline_rolls, gt_rolls)
print_dataset_metrics(baseline_metrics)

results = {
    'model_type': MODEL_TYPE,
    'run_timestamp': RUN_TIMESTAMP,
    'temperature': GENERATION_TEMPERATURE,
    'top_k': GENERATION_TOP_K,
    'model': model_metrics,
    'baseline': baseline_metrics,
}
out_path = RUN_DIR / f'eval_results_{MODEL_TYPE}.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved eval results → {out_path}')

# Keep last sample in scope for the piano-roll visualization below
model_roll = gen_rolls[-1]
prefix_ids = extract_prefix(test_samples[-1][0], tokenizer)
cont_ids   = generate_conditioned(model, prefix_ids, OPTION2_CONT_MAX_LEN, DEVICE,
                                  temperature=GENERATION_TEMPERATURE, top_k=GENERATION_TOP_K)

In [ ]:
import matplotlib.pyplot as plt

prefix_roll = tokens_to_pianoroll(prefix_ids.tolist(), tokenizer,
                                  duration_seconds=OPTION2_PREFIX_SECONDS)
combined = np.concatenate([prefix_roll, model_roll], axis=0).T

split_frame = len(prefix_roll)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(combined, aspect='auto', origin='lower', cmap='Blues', interpolation='nearest')
ax.axvline(split_frame - 0.5, color='red', linewidth=1.5, label='prefix | generated')
ax.set_xlabel('Frame (40 fps)')
ax.set_ylabel('Pitch (MIDI 21–108)')
ax.set_title(f'Piano-roll: prefix + generated continuation  [{MODEL_TYPE}, temp={GENERATION_TEMPERATURE}, top_k={GENERATION_TOP_K}]')
ax.legend(loc='upper right')
plt.tight_layout()
out_img = RUN_DIR / f'pianoroll_{MODEL_TYPE}.png'
plt.savefig(out_img, dpi=150)
plt.show()
print('Piano-roll saved to', out_img)

## Done

Each training run creates its own timestamped folder under `MyDrive/CSE253/option2/` on Google Drive (or `logs/option2/` locally).

```
option2/
└── {MODEL_TYPE}_{YYYYMMDD_HHMMSS}/
    ├── option2_{type}_best.pt          ← best checkpoint
    ├── train_history.json              ← per-epoch loss log
    ├── loss_curve_{type}.png           ← train/val loss graph
    ├── symbolic_conditioned.mid        ← generated MIDI
    ├── eval_results_{type}.json        ← model vs baseline metrics
    └── pianoroll_{type}.png            ← piano-roll visualization
```

In [ ]:
print("commit test")